In [3]:
# Environment Setup
import os
import warnings
warnings.filterwarnings('ignore')

from dotenv import load_dotenv
load_dotenv()

# Verify API key
if os.getenv("GEMINI_API_KEY"):
    print("✅ GEMINI_API_KEY found")
else:
    print("❌ GEMINI_API_KEY not found - please set it in your .env file")

✅ GEMINI_API_KEY found


In [5]:
# Core Imports

# Standard library
import numpy as np
import pandas as pd
import asyncio
import json

# LangChain components
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings

# RAGAS components
from ragas import SingleTurnSample, EvaluationDataset, evaluate
from ragas.metrics import (
    Faithfulness,
    ResponseRelevancy,
    LLMContextPrecisionWithReference,
    LLMContextRecall,
    ContextEntityRecall,
    NoiseSensitivity
)
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper

print("✅ All imports successful")

✅ All imports successful


In [7]:
# Initialize base models (GEMINI)
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0)
embeddings = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001")

# Wrap for RAGAS compatibility
ragas_llm = LangchainLLMWrapper(llm)
ragas_embeddings = LangchainEmbeddingsWrapper(embeddings)

print("✅ LLM initialized:gemini-2.5-flash")
print("✅ Embeddings initialized: models/gemini-embedding-001")
print("✅ RAGAS wrappers ready")

✅ LLM initialized:gemini-2.5-flash
✅ Embeddings initialized: models/gemini-embedding-001
✅ RAGAS wrappers ready


In [8]:
# Helper function for running async code in Jupyter

def run_async(coro):
    """Helper to run async code in Jupyter notebooks"""
    try:
        loop = asyncio.get_event_loop()
        if loop.is_running():
            # We're in Jupyter with an existing loop
            import nest_asyncio
            nest_asyncio.apply()
            return loop.run_until_complete(coro)
        else:
            return asyncio.run(coro)
    except RuntimeError:
        return asyncio.run(coro)

print("✅ Async helper ready")

✅ Async helper ready


In [ ]:
# Define our test case

# The response we want to evaluate
test_response = "The first Super Bowl was held on January 15, 1967 in Los Angeles. It was a sunny day with clear skies."

# The context that was retrieved (source of truth)
test_context = [
    "The First AFL-NFL World Championship Game was played on December 20, 2024, at the Los Angeles Memorial Coliseum in Newyork."
]

print("📝 Response to evaluate:")
print(f"   '{test_response}'")
print("\n📚 Retrieved context:")
print(f"   '{test_context[0]}'")

📝 Response to evaluate:
   'The first Super Bowl was held on January 15, 1967 in Los Angeles. It was a sunny day with clear skies.'

📚 Retrieved context:
   'The First AFL-NFL World Championship Game was played on January 15, 1967, at the Los Angeles Memorial Coliseum in Los Angeles, California.'


In [11]:
# Run actual RAGAS Faithfulness metric

# Create sample in RAGAS format
faithfulness_sample = SingleTurnSample(
    user_input="When was the first Super Bowl?",
    response=test_response,
    retrieved_contexts=test_context
)

# Initialize and run the metric
faithfulness_metric = Faithfulness(llm=ragas_llm)

ragas_faithfulness = run_async(faithfulness_metric.single_turn_ascore(faithfulness_sample))

print("🔬 RAGAS Faithfulness Result")
print("=" * 50)
print(f"   RAGAS metric score:  {ragas_faithfulness:.2f}")

🔬 RAGAS Faithfulness Result
   RAGAS metric score:  0.50
